# Stock Price Prediction
## Final Project

Predicting the next-day closing price of Apple (AAPL) stock using two models:
1. Linear Regression with lagged features
2. LSTM neural network

Data comes from Yahoo Finance via yfinance. We evaluate using MAE, RMSE, and R².


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

## 1. Data Collection

We pull 3 years of daily AAPL data from Yahoo Finance. The columns we get are Open, High, Low, Close, and Volume.


In [ ]:
from datetime import date, timedelta

TODAY      = date.today()
START_DATE = TODAY - timedelta(days=3 * 365)

data = yf.download('AAPL',
                   start=START_DATE.strftime('%Y-%m-%d'),
                   end=TODAY.strftime('%Y-%m-%d'),
                   auto_adjust=True, progress=False)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.droplevel(1)

df_raw = data.reset_index()
df_raw.columns = [c.strip() for c in df_raw.columns]

print(f'Shape      : {df_raw.shape}')
print(f'Date range : {df_raw["Date"].min().date()} to {df_raw["Date"].max().date()}')
df_raw.head()


## 2. Data Description & Preprocessing

The dataset has about 756 rows of daily stock data. Each row has the date, open/high/low/close prices, and trading volume.

For cleaning, we drop any missing rows, remove outliers using the IQR method on the Close column, convert everything to numeric, and add a simple Up/Down label for the EDA plots.


In [ ]:
print('Missing values per column:')
print(df_raw.isnull().sum())
df_raw = df_raw.dropna()

q1 = df_raw['Close'].quantile(0.25)
q3 = df_raw['Close'].quantile(0.75)
iqr = q3 - q1
low_bound  = q1 - 1.5 * iqr
high_bound = q3 + 1.5 * iqr
before = len(df_raw)
df_raw = df_raw[(df_raw['Close'] >= low_bound) & (df_raw['Close'] <= high_bound)].copy()
print(f'Outlier removal: {before} -> {len(df_raw)} rows (removed {before - len(df_raw)})')

df_raw['dir'] = (df_raw['Close'] > df_raw['Close'].shift(1)).map({True: 'Up', False: 'Down'})
df_raw = df_raw.dropna()

for col in ['Open','High','Low','Close','Volume']:
    df_raw[col] = pd.to_numeric(df_raw[col])

print('Descriptive Statistics:')
df_raw[['Open','High','Low','Close','Volume']].describe().round(2)


## 3. Feature Engineering

We add some common technical indicators on top of the raw price data: RSI (14-day), a 20-day simple moving average, 30 and 90-day moving averages, exponential moving averages (12 and 26-day), MACD, and normalized volume.


In [ ]:
df = df_raw[['Date','Open','High','Low','Close','Volume']].copy().set_index('Date')

delta = df['Close'].diff()
gain  = delta.where(delta > 0, 0).ewm(alpha=1/14, adjust=False).mean()
loss  = (-delta.where(delta < 0, 0)).ewm(alpha=1/14, adjust=False).mean()
df['RSI']     = 100 - (100 / (1 + gain / loss))

df['SMA_20']  = df['Close'].rolling(window=20).mean()
df['MA30']    = df['Close'].rolling(30).mean()
df['MA90']    = df['Close'].rolling(90).mean()

df['EMA_12']  = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26']  = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD']    = df['EMA_12'] - df['EMA_26']
df['Vol_norm'] = df['Volume'] / df['Volume'].rolling(20).mean()

df = df.dropna()
print(f'Dataset shape after feature engineering: {df.shape}')
df[['Close','RSI','SMA_20','MACD','Vol_norm']].tail()


## 4. Exploratory Data Analysis (EDA)

A few charts to get a feel for the data before we start modeling.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('AAPL Stock - Exploratory Data Analysis', fontsize=15, fontweight='bold')

axes[0, 0].plot(df.index, df['Close'], color='steelblue', linewidth=1)
axes[0, 0].set_title('Closing Price Over Time')
axes[0, 0].set_xlabel('Date'); axes[0, 0].set_ylabel('Price (USD)')
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

axes[0, 1].hist(df['Close'], bins=40, color='coral', edgecolor='white')
axes[0, 1].set_title('Distribution of Close Price')
axes[0, 1].set_xlabel('Price (USD)'); axes[0, 1].set_ylabel('Frequency')

axes[0, 2].boxplot(df['Close'], patch_artist=True, boxprops=dict(facecolor='lightgreen'))
axes[0, 2].set_title('Boxplot - Close Price')
axes[0, 2].set_ylabel('Price (USD)')

axes[1, 0].bar(df.index, df['Volume'], color='orchid', width=1)
axes[1, 0].set_title('Daily Volume')
axes[1, 0].set_xlabel('Date'); axes[1, 0].set_ylabel('Shares')
axes[1, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

counts = df_raw['dir'].value_counts()
axes[1, 1].bar(counts.index, counts.values, color=['limegreen','tomato'])
axes[1, 1].set_title('Price Direction (Up / Down)')
axes[1, 1].set_ylabel('Days')

axes[1, 2].scatter(df['Open'], df['Close'], alpha=0.4, s=10, color='navy')
axes[1, 2].set_title('Open vs Close Price')
axes[1, 2].set_xlabel('Open (USD)'); axes[1, 2].set_ylabel('Close (USD)')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr = df[['Open','High','Low','Close','Volume']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5, ax=axes[0])
axes[0].set_title('Correlation Heatmap')

axes[1].plot(df.index, df['Close'], label='Close', alpha=0.5, linewidth=1)
axes[1].plot(df.index, df['MA30'], label='30-day MA', linewidth=1.5)
axes[1].plot(df.index, df['MA90'], label='90-day MA', linewidth=1.5)
axes[1].set_title('Close Price with Moving Averages')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df.index, df['Close'], color='steelblue', linewidth=1.2, label='Close')
axes[0].set_title('AAPL - Technical Indicators Overview')
axes[0].set_ylabel('Price (USD)'); axes[0].legend()

axes[1].plot(df.index, df['RSI'], color='darkorange', linewidth=1)
axes[1].axhline(70, color='red', linestyle='--', linewidth=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
axes[1].set_ylabel('RSI'); axes[1].legend()

axes[2].bar(df.index, df['MACD'],
            color=np.where(df['MACD'] >= 0, 'limegreen', 'tomato'), width=1, label='MACD')
axes[2].set_ylabel('MACD'); axes[2].set_xlabel('Date'); axes[2].legend()
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.tight_layout()
plt.show()


## 5. Model 1 - Linear Regression

For each day, we take the past 10 days of Close, RSI, MACD, and SMA_20 as input features (so 40 features total) and try to predict the next day's Close price. The data is split 80/20 in time order.


In [ ]:
LAG_DAYS = 10

def build_lag_features(dataframe, lags=10):
    lag_df = pd.DataFrame(index=dataframe.index)
    for i in range(1, lags + 1):
        lag_df[f'Close_lag{i}'] = dataframe['Close'].shift(i)
        lag_df[f'RSI_lag{i}']   = dataframe['RSI'].shift(i)
        lag_df[f'MACD_lag{i}']  = dataframe['MACD'].shift(i)
        lag_df[f'SMA20_lag{i}'] = dataframe['SMA_20'].shift(i)
    lag_df['target'] = dataframe['Close']
    return lag_df.dropna()

lag_df = build_lag_features(df, lags=LAG_DAYS)
X_lag = lag_df.drop(columns='target').values
y_lag = lag_df['target'].values

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lag, y_lag, test_size=0.2, random_state=42, shuffle=False)

print(f'Train size : {len(X_train_lr)} samples')
print(f'Test size  : {len(X_test_lr)} samples')
print(f'Features   : {X_lag.shape[1]}')

lr = LinearRegression()
lr.fit(X_train_lr, y_train_lr)
lr_pred_test = lr.predict(X_test_lr)
print('Linear Regression model trained successfully.')


## 6. Model Evaluation

We use three metrics to see how well the models do: MAE (average error in dollars), RMSE (similar but penalizes big misses more), and R² (how much of the price variation the model explains).


In [ ]:
def regression_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{'='*45}')
    print(f'  {label}')
    print(f'{'='*45}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  R2   : {r2:.6f}')
    print()
    return mae, rmse, r2

lr_mae, lr_rmse, lr_r2 = regression_metrics(y_test_lr, lr_pred_test, 'Linear Regression - Test Set')


In [ ]:
test_dates_lr = lag_df.index[-len(y_test_lr):]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(test_dates_lr, y_test_lr, label='Actual', linewidth=1.2, color='steelblue')
axes[0].plot(test_dates_lr, lr_pred_test, label='Predicted', linewidth=1.2,
             linestyle='--', color='tomato')
axes[0].set_title('Linear Regression - Actual vs Predicted (Test Set)')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Close Price (USD)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

residuals_lr = y_test_lr - lr_pred_test
axes[1].scatter(lr_pred_test, residuals_lr, alpha=0.5, s=15, color='tomato')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Linear Regression - Residual Plot')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()


In [ ]:
lr_last_date = df.index[-1]
close_buf = list(df['Close'].values[-LAG_DAYS:])
rsi_buf   = list(df['RSI'].values[-LAG_DAYS:])
macd_buf  = list(df['MACD'].values[-LAG_DAYS:])
sma_buf   = list(df['SMA_20'].values[-LAG_DAYS:])

lr_future_prices = []
lr_future_dates  = list(pd.bdate_range(start=lr_last_date + pd.Timedelta(days=1), periods=5))

print('LR 5-Day Forecast:')
for i in range(5):
    row = []
    for lag in range(1, LAG_DAYS + 1):
        row += [close_buf[-lag], rsi_buf[-lag], macd_buf[-lag], sma_buf[-lag]]
    pred = lr.predict([row])[0]
    lr_future_prices.append(float(pred))
    print(f'  Day {i+1} ({lr_future_dates[i].date()}): ${pred:.2f}')
    close_buf.append(pred); rsi_buf.append(rsi_buf[-1])
    macd_buf.append(macd_buf[-1]); sma_buf.append(sma_buf[-1])

lr_plot_hist  = df['Close'].iloc[-120:]
lr_bridge_idx = [lr_last_date] + lr_future_dates
lr_bridge_val = [float(df['Close'].iloc[-1])] + lr_future_prices

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(lr_plot_hist.index, lr_plot_hist.values,
        color='steelblue', linewidth=1.5, label='Historical Close')
ax.plot(lr_bridge_idx, lr_bridge_val, color='tomato', linewidth=2,
        linestyle='--', marker='o', markersize=6, label='LR 5-Day Forecast')
for d, p in zip(lr_future_dates, lr_future_prices):
    ax.annotate(f'${p:.2f}', xy=(d, p), xytext=(5, 8),
                textcoords='offset points', fontsize=8, color='tomato')
ax.axvline(lr_last_date, color='gray', linestyle=':', linewidth=1)
ax.set_title('AAPL - Linear Regression Autoregressive 5-Day Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## 7. Model 2 - LSTM Neural Network

The LSTM takes 8 features (Close, High, Low, RSI, SMA_20, EMA_12, MACD, Vol_norm) over a 90-day window to predict the next day's close.

Architecture: 3 LSTM layers (256, 128, 64 units) with dropout, then 2 dense layers down to 1 output. Trained with Huber loss and Adam optimizer, with early stopping and learning rate reduction.

Same 80/20 time-ordered split. 10% of training data used for validation.


In [ ]:
feature_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler  = MinMaxScaler(feature_range=(0, 1))

LSTM_FEATURES = ['Close','High','Low','RSI','SMA_20','EMA_12','MACD','Vol_norm']

scaled_features = feature_scaler.fit_transform(df[LSTM_FEATURES].values)
scaled_target   = target_scaler.fit_transform(df[['Close']].values)

WINDOW = 90

def make_sequences_multivariate(features, target, window):
    X, y = [], []
    for i in range(window, len(features)):
        X.append(features[i - window:i])
        y.append(target[i, 0])
    return np.array(X), np.array(y)

X_seq, y_seq = make_sequences_multivariate(scaled_features, scaled_target, WINDOW)

split = int(len(X_seq) * 0.8)
X_tr, X_te = X_seq[:split], X_seq[split:]
y_tr, y_te = y_seq[:split], y_seq[split:]

print(f'LSTM Train shape : {X_tr.shape}')
print(f'LSTM Test shape  : {X_te.shape}')


In [ ]:
model = Sequential([
    LSTM(256, return_sequences=True, input_shape=(WINDOW, X_tr.shape[2])),
    Dropout(0.1),
    LSTM(128, return_sequences=True),
    Dropout(0.1),
    LSTM(64, return_sequences=False),
    Dropout(0.1),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
model.compile(optimizer=Adam(learning_rate=0.001), loss='huber')
model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)

history = model.fit(
    X_tr, y_tr,
    validation_split=0.1,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('LSTM Training & Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Huber Loss')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
lstm_pred_scaled = model.predict(X_te)
lstm_pred  = target_scaler.inverse_transform(lstm_pred_scaled)
y_te_inv   = target_scaler.inverse_transform(y_te.reshape(-1, 1))

lstm_mae, lstm_rmse, lstm_r2 = regression_metrics(
    y_te_inv.flatten(), lstm_pred.flatten(), 'LSTM Neural Network - Test Set')

test_dates_lstm = df.index[WINDOW + split:]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(test_dates_lstm, y_te_inv, label='Actual', linewidth=1.2, color='steelblue')
axes[0].plot(test_dates_lstm, lstm_pred, label='Predicted', linewidth=1.2,
             linestyle='--', color='darkorange')
axes[0].set_title('LSTM - Actual vs Predicted (Test Set)')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('Close Price (USD)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

residuals_lstm = y_te_inv.flatten() - lstm_pred.flatten()
axes[1].scatter(lstm_pred, residuals_lstm, alpha=0.5, s=15, color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('LSTM - Residual Plot')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()


## 8. LSTM - 5-Day Forecast


In [ ]:
last_window = scaled_features[-WINDOW:].copy()
current_sequence = last_window.copy()
future_predictions_scaled = []

for day in range(5):
    X_input = np.array([current_sequence])
    pred_scaled = model.predict(X_input, verbose=0)
    future_predictions_scaled.append(pred_scaled[0][0])
    new_step = current_sequence[-1:].copy()
    new_step[0, 0] = pred_scaled[0][0]
    current_sequence = np.vstack([current_sequence[1:], new_step])

future_predictions = target_scaler.inverse_transform(
    np.array(future_predictions_scaled).reshape(-1, 1))
predicted_values = [float(p[0]) for p in future_predictions]

current_price = float(df['Close'].iloc[-1])
current_rsi   = float(df['RSI'].iloc[-1])
current_sma   = float(df['SMA_20'].iloc[-1])
final_price   = predicted_values[-1]
price_diff    = final_price - current_price

print(f'Current Price : ${current_price:.2f}')
print(f'Current RSI   : {current_rsi:.1f}')
print(f'Current SMA20 : ${current_sma:.2f}')
print()

if price_diff > 0:
    print(f'[Trend] UPWARD movement expected (+${price_diff:.2f} over 5 days).')
else:
    print(f'[Trend] DOWNWARD movement expected (-${abs(price_diff):.2f} over 5 days).')

if current_price > current_sma:
    print(f'[SMA-20] Price above 20-day average - positive signal.')
else:
    print(f'[SMA-20] Price below 20-day average - negative signal.')

if current_rsi > 70:
    print(f'[RSI {current_rsi:.1f}] OVERBOUGHT - potential pullback anticipated.')
elif current_rsi < 30:
    print(f'[RSI {current_rsi:.1f}] OVERSOLD - potential bounce anticipated.')
else:
    print(f'[RSI {current_rsi:.1f}] Neutral zone - model relies on learned patterns.')

last_date = df.index[-1]
future_dates_forecast = list(pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=5))
print('Day-by-Day LSTM Forecast:')
for i, price in enumerate(predicted_values, 1):
    print(f'  Day {i} ({future_dates_forecast[i-1].date()}): ${price:.2f}')


In [ ]:
plot_historical = df['Close'].iloc[-120:]
bridge_index    = [last_date] + future_dates_forecast
bridge_values   = [current_price] + predicted_values

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(plot_historical.index, plot_historical.values,
        color='steelblue', linewidth=1.5, label='Historical Close')
ax.plot(bridge_index, bridge_values, color='tomato', linewidth=2,
        linestyle='--', marker='o', markersize=6, label='LSTM 5-Day Forecast')
for date, price in zip(future_dates_forecast, predicted_values):
    ax.annotate(f'${price:.2f}', xy=(date, price), xytext=(5, 8),
                textcoords='offset points', fontsize=8, color='tomato')
ax.axvline(last_date, color='gray', linestyle=':', linewidth=1)
ax.set_title('AAPL - Historical Close Price vs 5-Day LSTM Forecast',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()


## 9. Model Comparison & Conclusion


In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'LSTM Neural Network'],
    'MAE'  : [lr_mae,   lstm_mae],
    'RMSE' : [lr_rmse,  lstm_rmse],
    'R2'   : [lr_r2,    lstm_r2],
}).set_index('Model')

print('Model Comparison Table:')
print(results.to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = ['MAE', 'RMSE', 'R2']
colors  = ['#2196F3', '#FF5722']

for i, m in enumerate(metrics):
    vals = [results.loc['Linear Regression', m], results.loc['LSTM Neural Network', m]]
    bars = axes[i].bar(['LR', 'LSTM'], vals, color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(m)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                     f'{v:.4f}', ha='center', va='bottom', fontsize=10)
    if m == 'R2':
        axes[i].set_ylim(min(vals) - 0.05, 1.01)

fig.suptitle('Model Performance Comparison - AAPL Close Price Prediction',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Conclusion:')
if lstm_r2 > lr_r2:
    print('  LSTM Neural Network outperforms Linear Regression on all metrics.')
    print('  LSTM better captures the non-linear temporal dependencies in stock price data.')
else:
    print('  Linear Regression is competitive; consider dataset size / training time tradeoffs.')
